# 🎧 감성·발화스타일 음성합성 데이터셋 — 화자 중심 청음 노트북

`reciter_id`(화자 ID)를 고르면 그 화자의 오디오 샘플을 **텍스트·감정·스타일과 함께 재생**합니다.

| | |
|---|---|
| **데이터셋** | AI Hub 「감성 및 발화스타일 동시 고려 음성합성 데이터」 (datasetkey=71349) |
| **필요 패키지** | `pandas`, `ipywidgets` (8.x), `IPython` — 오디오 라이브러리 불필요 |
| **커널** | `porte` (또는 ipywidgets 8.x가 설치된 아무 커널) |

**시작 전 확인**
1. 아래 **설정 셀의 `DATASET_ROOT`** 가 본인 경로인지 — 이 노트북에서 고칠 곳은 여기 한 곳뿐입니다
2. `meta/metadata.csv` 가 있어야 합니다 (없다면 `preprocess/build_metadata.py` 먼저 실행)

**구성**
1. 설정 & 메타데이터 로드 → 2. 절대경로 헬퍼 → 3. 화자 정보 카드 → 4. 샘플 청취 함수
→ 5. 필터 조합 예시 → 6. **인터랙티브 위젯(권장)** → 7. 화자별 CSV → 8. 데이터 검증

> ⚠️ 오디오 위젯은 wav를 노트북에 base64로 내장합니다. 많이 재생한 뒤 저장하면 파일이 수십 MB로
> 불어나니, 커밋 전에 **Kernel → Restart & Clear Output** 을 한 번 눌러 주세요.

## 1. 설정 & 메타데이터 로드

In [ ]:
# ============================================================
# ⚙️ 설정 — 환경이 바뀌면 DATASET_ROOT 만 고치면 됩니다
# ============================================================
from pathlib import Path
import pandas as pd
from IPython.display import Audio, display, HTML

# data/ 와 meta/ 를 담고 있는 데이터셋 루트
DATASET_ROOT = Path("/home/work/my-datasets/kor_senti_style_tts_datasets")

META_DIR        = DATASET_ROOT / "meta"
META_PATH       = META_DIR / "metadata.csv"
PER_SPEAKER_DIR = META_DIR / "metadatas_per_speaker"

# --- 사전 점검: 경로가 틀리면 여기서 이유를 알려주고 멈춥니다 ---
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"DATASET_ROOT 없음: {DATASET_ROOT}\n"
        "  → data/ 와 meta/ 가 들어 있는 폴더 경로로 고쳐 주세요."
    )
if not META_PATH.is_file():
    raise FileNotFoundError(
        f"metadata.csv 없음: {META_PATH}\n"
        "  → 메타데이터를 먼저 생성하세요:\n"
        f"     python3 preprocess/build_metadata.py \\\n"
        f"       --data-dir {DATASET_ROOT}/data \\\n"
        f"       --base-dir {DATASET_ROOT} \\\n"
        f"       --output-dir {META_DIR} --use-index"
    )

# --- 로드 ---------------------------------------------------
# 전체 32컬럼: 약 8.5초 / 1.4GiB.
# 메모리를 아끼려면 아래 USECOLS 주석을 풀어 17컬럼(0.7GiB)만 읽으세요.
USECOLS = None
# USECOLS = ['file_id','reciter_id','reciter_age','reciter_gender','style','sub_style',
#            'emotion','intensity','duration','duration_valid','text_tr','text_ptr',
#            'base_dir','audio_path','audio_exists','split','votes_avg']

df = pd.read_csv(META_PATH, usecols=USECOLS, low_memory=False)

print(f"메타데이터  : {META_PATH}")
print(f"총 row 수   : {len(df):,}")
print(f"화자 수     : {df['reciter_id'].nunique()}명")
print(f"split 분포  : {df['split'].value_counts().to_dict()}")
print(f"총 duration : {df['duration'].sum()/3600:,.1f}시간")
print(f"화자 ID     : {sorted(df['reciter_id'].dropna().unique().astype(int))[:20]} ...")
df.head(3)

## 2. 절대 경로 헬퍼

`base_dir + audio_path` 결합. 서버 이전 시 base_dir만 갈아끼우면 됨.

In [ ]:
def get_audio_abspath(row):
    """한 row → wav 절대 경로 (Path 객체)."""
    if pd.isna(row.get('audio_path')) or not row['audio_path']:
        return None
    return Path(row['base_dir']) / row['audio_path']


# 동작 확인
sample_row = df.iloc[0]
abs_path = get_audio_abspath(sample_row)
print(f"base_dir   : {sample_row['base_dir']}")
print(f"audio_path : {sample_row['audio_path']}")
print(f"절대 경로  : {abs_path}")
print(f"실제 존재  : {abs_path.exists() if abs_path else False}")

## 3. 화자 정보 출력 함수

In [ ]:
def show_speaker_info(df, speaker_id):
    """화자 한 명의 메타 정보를 카드 형태로 출력."""
    sub = df[df['reciter_id'] == speaker_id]
    if len(sub) == 0:
        display(HTML(f"<p style='color:red'>화자 {speaker_id} 없음</p>"))
        return None
    info = sub.iloc[0]
    total_min = sub['duration'].sum() / 60
    
    html = f"""
    <div style='border:1px solid #ddd; padding:14px 18px; border-radius:8px; background:#f9fbfd; margin:8px 0;'>
      <h3 style='margin:0 0 10px 0; color:#1565C0;'>🎤 화자 {int(speaker_id)}</h3>
      <table style='border-collapse:collapse;'>
        <tr><td style='padding:2px 12px 2px 0;'><b>성별</b></td><td>{info['reciter_gender']}</td></tr>
        <tr><td style='padding:2px 12px 2px 0;'><b>나이</b></td><td>{info['reciter_age']}세</td></tr>
        <tr><td style='padding:2px 12px 2px 0;'><b>총 발화 수</b></td><td>{len(sub):,}건</td></tr>
        <tr><td style='padding:2px 12px 2px 0;'><b>총 duration</b></td><td>{total_min:.1f}분</td></tr>
        <tr><td style='padding:2px 12px 2px 0;'><b>평균 duration</b></td><td>{sub['duration'].mean():.2f}초</td></tr>
      </table>
    </div>
    """
    display(HTML(html))
    
    # 분포 표
    style_df = sub['style'].value_counts().to_frame('발화 수')
    emotion_df = sub['emotion'].value_counts().to_frame('발화 수')
    
    print("발화 스타일 분포:")
    print(style_df.to_string())
    print()
    print("감정 분포:")
    print(emotion_df.to_string())
    
    return sub


# 사용 예시
_ = show_speaker_info(df, speaker_id=9)

## 4. 샘플 청취 함수

화자 + 추가 필터(스타일·감정·강도)로 샘플 검색·재생.

In [ ]:
def play_samples_for_speaker(
    df,
    speaker_id,
    n=5,
    style=None,
    emotion=None,
    sub_style=None,
    intensity=None,
    split=None,
    text_contains=None,
    valid_only=True,
    random_state=42,
):
    """
    특정 화자의 샘플을 필터링 후 재생.
    각 샘플마다 file_id, 스타일/감정, duration, tr/ptr, audio_path, 오디오 위젯 표시.

    split : 'train' | 'valid' | None(전체)
    """
    sub = df[df['reciter_id'] == speaker_id]
    if len(sub) == 0:
        display(HTML(f"<p style='color:red'>화자 {speaker_id} 없음</p>"))
        return

    if valid_only:
        sub = sub[sub['audio_exists'] & sub['duration_valid']]
    if style:         sub = sub[sub['style'] == style]
    if sub_style:     sub = sub[sub['sub_style'] == sub_style]
    if emotion:       sub = sub[sub['emotion'] == emotion]
    if intensity:     sub = sub[sub['intensity'] == intensity]
    if split:         sub = sub[sub['split'] == split]
    if text_contains: sub = sub[sub['text_tr'].str.contains(text_contains, na=False)]

    if len(sub) == 0:
        display(HTML("<p style='color:gray;'>조건에 맞는 샘플 없음</p>"))
        return

    display(HTML(f"<p><b>조건 매칭:</b> {len(sub):,}건 / 재생: 최대 {min(n, len(sub))}건</p>"))

    samples = sub.sample(min(n, len(sub)), random_state=random_state)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        ptr_html = f"<p style='color:#666; font-size:0.9em; margin:4px 0;'><b>ptr:</b> {row['text_ptr']}</p>" \
                   if pd.notna(row['text_ptr']) and row['text_ptr'] != row['text_tr'] else ""

        info_html = f"""
        <div style='border-left:3px solid #2196F3; padding:10px 14px; margin:10px 0; background:#fafcff;'>
          <p style='margin:0 0 6px 0;'>
            <b>#{i}  {row['file_id']}</b> &nbsp;|&nbsp;
            <span style='color:#1976D2;'>{row['style']}/{row['sub_style']}</span> &nbsp;|&nbsp;
            <span style='color:#C2185B;'>{row['emotion']}(강도 {row['intensity']})</span> &nbsp;|&nbsp;
            <span style='color:#666;'>{row['duration']:.2f}초</span> &nbsp;|&nbsp;
            <span style='color:#388E3C;'>{row['split']}</span>
          </p>
          <p style='margin:4px 0;'><b>tr:</b> {row['text_tr']}</p>
          {ptr_html}
          <p style='color:#999; font-size:0.8em; margin:4px 0;'>📁 {row['audio_path']}</p>
        </div>
        """
        display(HTML(info_html))

        audio_path = get_audio_abspath(row)
        if audio_path and audio_path.exists():
            display(Audio(str(audio_path)))
        else:
            display(HTML(f"<p style='color:red;'>⚠ wav 파일 없음: {audio_path}</p>"))


# 사용 예시
play_samples_for_speaker(df, speaker_id=9, n=3)

## 5. 다양한 필터 조합 예시

In [ ]:
# 5-1. 화자 9번의 분노 감정 발화
play_samples_for_speaker(df, speaker_id=9, emotion="분노", n=3)

In [ ]:
# 5-2. 화자 9번의 애니체 + 강도 3 (최고 강도)
play_samples_for_speaker(df, speaker_id=9, style="애니체", intensity=3, n=2)

In [ ]:
# 5-3. 화자 9번의 '엄마' 단어 포함 발화
play_samples_for_speaker(df, speaker_id=9, text_contains="엄마", n=3)

## 6. 인터랙티브 위젯 (드롭다운으로 화자 선택)

화자 ID를 선택하면 정보가 자동 표시되고, 추가 필터로 검색·청취 가능.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

# 옵션 자동 생성
speakers_avail    = sorted(df['reciter_id'].dropna().unique().astype(int))
styles_avail      = ["(전체)"] + sorted(df['style'].dropna().unique().tolist())
emotions_avail    = ["(전체)"] + sorted(df['emotion'].dropna().unique().tolist())
intensities_avail = ["(전체)"] + sorted([int(x) for x in df['intensity'].dropna().unique()])
splits_avail      = ["(전체)"] + sorted(df['split'].dropna().unique().tolist())

_st = {'description_width': 'initial'}
w_speaker   = widgets.Dropdown(options=speakers_avail, description="화자 ID:", style=_st)
w_style     = widgets.Dropdown(options=styles_avail, description="발화체:", style=_st)
w_emotion   = widgets.Dropdown(options=emotions_avail, description="감정:", style=_st)
w_intensity = widgets.Dropdown(options=intensities_avail, description="강도:", style=_st)
w_split     = widgets.Dropdown(options=splits_avail, description="split:", style=_st)
w_text      = widgets.Text(value="", placeholder="단어 검색 (선택)", description="텍스트:", style=_st)
w_n         = widgets.IntSlider(min=1, max=15, value=5, description="샘플 수:", style=_st)
w_seed      = widgets.IntText(value=42, description="seed:", style=_st)
w_btn       = widgets.Button(description="🔍 검색 + 재생", button_style="primary",
                             layout=widgets.Layout(width='200px'))
w_info_btn  = widgets.Button(description="ℹ️ 화자 정보만", button_style="info",
                             layout=widgets.Layout(width='200px'))
w_out       = widgets.Output()


def on_search(_):
    with w_out:
        clear_output()
        show_speaker_info(df, w_speaker.value)
        kwargs = {"n": w_n.value, "random_state": w_seed.value}
        if w_style.value != "(전체)":     kwargs["style"] = w_style.value
        if w_emotion.value != "(전체)":   kwargs["emotion"] = w_emotion.value
        if w_intensity.value != "(전체)": kwargs["intensity"] = w_intensity.value
        if w_split.value != "(전체)":     kwargs["split"] = w_split.value
        if w_text.value.strip():          kwargs["text_contains"] = w_text.value.strip()
        print()
        play_samples_for_speaker(df, w_speaker.value, **kwargs)


def on_info(_):
    with w_out:
        clear_output()
        show_speaker_info(df, w_speaker.value)


w_btn.on_click(on_search)
w_info_btn.on_click(on_info)

display(widgets.VBox([
    widgets.HBox([w_speaker, w_style, w_emotion, w_intensity]),
    widgets.HBox([w_split, w_n, w_seed]),
    widgets.HBox([w_text]),
    widgets.HBox([w_info_btn, w_btn]),
    w_out
]))

## 7. 화자별 CSV 직접 로드 (선택)

`metadatas_per_speaker/speaker_009.csv` 처럼 화자 단위 CSV를 직접 로드해서 작업하고 싶을 때.

In [ ]:
SPEAKER_ID = 9

per_speaker_csv = PER_SPEAKER_DIR / f"speaker_{SPEAKER_ID:03d}.csv"
print(f"로드: {per_speaker_csv}")

if per_speaker_csv.exists():
    sp_df = pd.read_csv(per_speaker_csv, low_memory=False)
    print(f"  → {len(sp_df):,}건")
    print()
    print("이 화자의 발화체별 평균 duration:")
    print(sp_df.groupby('style')['duration'].agg(['count', 'mean', 'sum']).round(2))
else:
    print(f"파일 없음 — PER_SPEAKER_DIR 경로와 화자 ID를 확인하세요.")
    print(f"  사용 가능한 화자: {sorted(int(p.stem.split('_')[1]) for p in PER_SPEAKER_DIR.glob('speaker_*.csv'))[:15]} ...")

## 8. (선택) 데이터 검증 체크리스트

In [ ]:
# 학습 가능한 데이터만 필터링
df_clean = df[
    df["audio_exists"] &
    df["duration_valid"] &
    df["text_tr"].notna() &
    df["duration"].between(0.5, 20)
].reset_index(drop=True)

print(f"전체 row    : {len(df):,}")
print(f"학습 가능   : {len(df_clean):,} ({len(df_clean)/len(df)*100:.2f}%)")
print()
print("--- 필터링 단계별 ---")
print(f"  audio_exists=True       : {df['audio_exists'].sum():,}")
print(f"  duration_valid=True     : {df['duration_valid'].sum():,}")
print(f"  text_tr 존재            : {df['text_tr'].notna().sum():,}")
print(f"  duration 0.5~20초       : {df['duration'].between(0.5, 20).sum():,}")